In [1]:
########################### Validate the accuracy of map time series using independent point samples ########

from datetime import datetime, timedelta
import os
import numpy as np
import datetime as dt
import matplotlib.pyplot as plt
import requests
import sys
import geopandas as gpd
import pandas as pd
from shapely.geometry import MultiPoint
from shapely.geometry import Point
import fiona
import contextily as ctx
from datetime import datetime
from sklearn.ensemble import RandomForestClassifier
import simplekml
from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from pyproj import Proj, transform
import ast
from pyproj import Transformer
import rasterio
from rasterio.features import geometry_mask
from rasterio.transform import from_origin
from rasterio.warp import calculate_default_transform, reproject
from rasterio.enums import Resampling

from scipy.ndimage import binary_fill_holes
from scipy.ndimage import convolve
from scipy.ndimage import distance_transform_edt
from affine import Affine
from scipy.spatial import KDTree
from rasterio.mask import mask
from rasterio.features import shapes
from shapely.geometry import shape
from simplekml import Kml
from osgeo import gdal
from rasterio.transform import rowcol
from sklearn.metrics import accuracy_score

In [2]:
def get_gdfutm(gdf):
    
    class_mapping = {'VZ': 1, 'WZ': 2, 'TZ': 3}
    if gdf.geometry.iloc[0].geom_type == "Point":
        # Extract latitude and longitude
        gdf["latitude"] = gdf.geometry.y
        gdf["longitude"] = gdf.geometry.x
        gdf["description"] = gdf["descriptio"]

        # Define the UTM Zone 17N EPSG code (WGS 84 / UTM Zone 17N)
        utm_17n_crs = "EPSG:32617"

        # Convert to UTM 17N
        gdf_utm = gdf.to_crs(utm_17n_crs)

        # Extract UTM coordinates
        gdf_utm["easting"] = gdf_utm.geometry.x
        gdf_utm["northing"] = gdf_utm.geometry.y

        # Print first few records
        print(gdf_utm[["latitude", "longitude", "easting", "northing"]].head())

        gdf["class_id"] = gdf["descriptio"].map(class_mapping)  # Apply mapping
        gdf_utm["description"] = gdf["descriptio"].values  # Copy values explicitly
        gdf_utm["class_id"] = gdf["class_id"].values  # Copy mapped values
    else:
        print("Shapefile does not contain Point geometries.")
    
    return gdf_utm

In [3]:
year = 2014
# Load the shapefile
shapefile_path = "/Users/bz294/Documents/WhiteZone/Points/KML/Validation/2014/class_2014_latlon.shp"
gdf = gpd.read_file(shapefile_path)
gdf_utm1 = get_gdfutm(gdf)

shapefile_path = "/Users/bz294/Documents/WhiteZone/Points/KML/Validation/2004/class_2004_latlon.shp"
gdf = gpd.read_file(shapefile_path)
gdf_utm2 = get_gdfutm(gdf)

gdf_utm = gpd.GeoDataFrame(pd.concat([gdf_utm1, gdf_utm2], ignore_index=True))

    latitude  longitude        easting      northing
0  25.207743 -80.673000  532941.913104  2.787991e+06
1  25.207810 -80.680361  532200.354640  2.787997e+06
2  25.220519 -80.680769  532155.887026  2.789404e+06
3  25.210318 -80.672327  533009.035922  2.788277e+06
4  25.222765 -80.665515  533691.805219  2.789657e+06
    latitude  longitude        easting      northing
0  25.320867 -80.406665  559717.543642  2.800610e+06
1  25.322498 -80.396104  560779.629282  2.800795e+06
2  25.315090 -80.407486  559637.733021  2.799970e+06
3  25.313274 -80.386207  561780.507951  2.799778e+06
4  25.311225 -80.386763  561725.569645  2.799551e+06


In [4]:
shapefile_paths = [
    "/Users/bz294/Documents/WhiteZone/Points/KML/Validation/2014/class_2014_latlon.shp",
    "/Users/bz294/Documents/WhiteZone/Points/KML/Validation/2010/class_2010_latlon.shp",
    "/Users/bz294/Documents/WhiteZone/Points/KML/Validation/2006/class_2006_latlon.shp",
    "/Users/bz294/Documents/WhiteZone/Points/KML/Validation/2004/class_2004_latlon.shp"
]

# Read and convert each shapefile, storing results in a list
gdf_utm_list = [get_gdfutm(gpd.read_file(path)) for path in shapefile_paths]

    latitude  longitude        easting      northing
0  25.207743 -80.673000  532941.913104  2.787991e+06
1  25.207810 -80.680361  532200.354640  2.787997e+06
2  25.220519 -80.680769  532155.887026  2.789404e+06
3  25.210318 -80.672327  533009.035922  2.788277e+06
4  25.222765 -80.665515  533691.805219  2.789657e+06
    latitude  longitude        easting      northing
0  25.245191 -80.503293  550023.073689  2.792190e+06
1  25.246638 -80.498876  550467.323876  2.792352e+06
2  25.245799 -80.497130  550643.485492  2.792260e+06
3  25.245469 -80.493846  550974.289741  2.792225e+06
4  25.245646 -80.487446  551618.851221  2.792247e+06
    latitude  longitude        easting      northing
0  25.250376 -80.469798  553394.130571  2.792777e+06
1  25.248443 -80.468467  553528.968259  2.792564e+06
2  25.244797 -80.468337  553543.662504  2.792160e+06
3  25.248480 -80.466147  553762.609614  2.792569e+06
4  25.250936 -80.469122  553461.967262  2.792840e+06
    latitude  longitude        easting      no

In [5]:
len(gdf_utm_list)

4

In [19]:
################## Read raster ###################################
years = [2014, 2010, 2006, 2004]

product_paths = [
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold07/CD_closest_GE/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold07/CD_previous_year/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold07/CD_previous_GE/"
]



for j in range(4): ## year
    file = "raster_" + str(years[j]) + "_cd.tif"
    print("year is:")
    print(years[j])
    for i in range(3): ## product
        print("product")
        print(i)
        print(product_paths[i])
        raster_path = product_paths[i] + file
        gdf_utm = gdf_utm_list[j]
        
        with rasterio.open(raster_path) as src:
            # Extract values at points
            values = list(src.sample(zip(gdf_utm["easting"], gdf_utm["northing"])))

        # Convert list of arrays to a flat NumPy array (assuming single-band raster)
        raster_values = np.array([v[0] for v in values])  # Extract first band
        # Add raster values to DataFrame
        
        gdf_utm["raster_value"] = raster_values

        # Compute accuracy (ignoring NaN values)
        valid_mask = ~np.isnan(raster_values) & gdf_utm["class_id"].notna()
        correct_predictions = (gdf_utm.loc[valid_mask, "class_id"] == gdf_utm.loc[valid_mask, "raster_value"]).sum()
        total_valid = valid_mask.sum()
        accuracy = (correct_predictions / total_valid) * 100 if total_valid > 0 else 0

        # Print results
        print(f"Current product is {i}")
        print(f"Overall Accuracy: {accuracy:.1f}%")
        #print(gdf_utm[["latitude", "longitude", "easting", "northing", "description", "class_id", "raster_value"]].head())

year is:
2014
product
0
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold07/CD_closest_GE/
Current product is 0
Overall Accuracy: 87.0%
product
1
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold07/CD_previous_year/
Current product is 1
Overall Accuracy: 87.0%
product
2
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold07/CD_previous_GE/
Current product is 2
Overall Accuracy: 87.0%
year is:
2010
product
0
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold07/CD_closest_GE/
Current product is 0
Overall Accuracy: 61.0%
product
1
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold07/CD_previous_year/
Current product is 1
Overall Accuracy: 61.0%
product
2
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold07/CD_previous_GE/
Current product is 2
Overall Accuracy: 61.0%
year is:
2006
product
0
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthresho

In [23]:
########### Re-org the above block by product first and then year for generateing the Table #####
################## Read raster ###################################
years = [2014, 2010, 2006, 2004]

product_paths = [
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_previous_GE/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_previous_year/"
]

for i in range(3):  # product
    accuracies = []  # Store accuracies for each year
    
    for j in range(4):  # year
        file = "raster_" + str(years[j]) + "_cd.tif"
        print("Year is:", years[j])
        print("Product:", i)
        print(product_paths[i])
        
        raster_path = product_paths[i] + file
        gdf_utm = gdf_utm_list[j]
        
        print(raster_path)
        with rasterio.open(raster_path) as src:
            # Extract values at points
            values = list(src.sample(zip(gdf_utm["easting"], gdf_utm["northing"])))

        # Convert list of arrays to a flat NumPy array (assuming single-band raster)
        raster_values = np.array([v[0] for v in values])
        gdf_utm["raster_value"] = raster_values

        # Compute accuracy (ignoring NaNs)
        valid_mask = ~np.isnan(raster_values) & gdf_utm["class_id"].notna()
        correct_predictions = (gdf_utm.loc[valid_mask, "class_id"] == gdf_utm.loc[valid_mask, "raster_value"]).sum()
        total_valid = valid_mask.sum()
        accuracy = (correct_predictions / total_valid) * 100 if total_valid > 0 else 0

        # Store accuracy for this year
        accuracies.append(accuracy)

        # Print per-year accuracy
        print(f"  Year {years[j]} Accuracy: {accuracy:.1f}%")
    
    # After looping through all years, compute average accuracy for this product
    avg_accuracy = np.mean(accuracies)
    print(f"===> Average Accuracy for Product {i}: {avg_accuracy:.1f}%\n")

Year is: 2014
Product: 0
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/raster_2014_cd.tif
  Year 2014 Accuracy: 86.3%
Year is: 2010
Product: 0
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/raster_2010_cd.tif
  Year 2010 Accuracy: 59.0%
Year is: 2006
Product: 0
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/raster_2006_cd.tif
  Year 2006 Accuracy: 77.9%
Year is: 2004
Product: 0
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/
/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/raster_2004_cd.tif
  Year 2004 Accuracy: 59

In [12]:
################## Read raster ###################################
product_paths = [
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_closest_GE/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_previous_GE/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold08/CD_previous_year/"
]

file = "raster_" + str(year) + "_cd.tif"

for i in range(3):
    raster_path = product_paths[i] + file

    with rasterio.open(raster_path) as src:
        # Extract values at points
        values = list(src.sample(zip(gdf_utm["easting"], gdf_utm["northing"])))

    # Convert list of arrays to a flat NumPy array (assuming single-band raster)
    raster_values = np.array([v[0] for v in values])  # Extract first band

    # Add raster values to DataFrame
    gdf_utm["raster_value"] = raster_values

    # Compute accuracy (ignoring NaN values)
    valid_mask = ~np.isnan(raster_values) & gdf_utm["class_id"].notna()
    correct_predictions = (gdf_utm.loc[valid_mask, "class_id"] == gdf_utm.loc[valid_mask, "raster_value"]).sum()
    total_valid = valid_mask.sum()
    accuracy = (correct_predictions / total_valid) * 100 if total_valid > 0 else 0

    # Print results
    print(f"Current product is {i}")
    print(f"Overall Accuracy: {accuracy:.2f}%")
    #print(gdf_utm[["latitude", "longitude", "easting", "northing", "description", "class_id", "raster_value"]].head())

Current product is 0
Overall Accuracy: 80.06%
Current product is 1
Overall Accuracy: 80.06%
Current product is 2
Overall Accuracy: 80.06%


In [ ]:
################## Read raster ###################################
product_paths = [
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold06/CD_closest_GE/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold06/CD_previous_GE/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold06/CD_previous_year/"
]

file = "raster_" + str(year) + "_cd.tif"

for i in range(3):
    raster_path = product_paths[i] + file

    with rasterio.open(raster_path) as src:
        # Extract values at points
        values = list(src.sample(zip(gdf_utm["easting"], gdf_utm["northing"])))

    # Convert list of arrays to a flat NumPy array (assuming single-band raster)
    raster_values = np.array([v[0] for v in values])  # Extract first band

    # Add raster values to DataFrame
    gdf_utm["raster_value"] = raster_values

    # Compute accuracy (ignoring NaN values)
    valid_mask = ~np.isnan(raster_values) & gdf_utm["class_id"].notna()
    correct_predictions = (gdf_utm.loc[valid_mask, "class_id"] == gdf_utm.loc[valid_mask, "raster_value"]).sum()
    total_valid = valid_mask.sum()
    accuracy = (correct_predictions / total_valid) * 100 if total_valid > 0 else 0

    # Print results
    print(f"Current product is {i}")
    print(f"Overall Accuracy: {accuracy:.2f}%")
    #print(gdf_utm[["latitude", "longitude", "easting", "northing", "description", "class_id", "raster_value"]].head())

In [ ]:
############# Investigate differences among the three datasets resulting from 0.6 ################
############# year by year ###############
product_paths = [
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold06/CD_closest_GE/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold06/CD_previous_GE/",
    "/Volumes/Malonelab/Research/Whitezone/Result/Emulation/Final/CDthreshold06/CD_previous_year/"
]

# Get the list of raster files from the first folder
raster_files = sorted([f for f in os.listdir(product_paths[0]) if f.endswith(".tif")])

# Store results
diff_results = {}

# Loop through each raster file
for raster_name in raster_files:
    raster_paths = [os.path.join(folder, raster_name) for folder in product_paths]

    # Open the three rasters
    with rasterio.open(raster_paths[0]) as src1, \
         rasterio.open(raster_paths[1]) as src2, \
         rasterio.open(raster_paths[2]) as src3:
        
        # Read raster data as numpy arrays
        raster1 = src1.read(1)  # First folder raster
        raster2 = src2.read(1)  # Second folder raster
        raster3 = src3.read(1)  # Third folder raster

        # Ensure all rasters have the same shape
        if not (raster1.shape == raster2.shape == raster3.shape):
            print(f"Shape mismatch for {raster_name}, skipping...")
            continue

        # Create a mask for differing pixels (excluding NoData values)
        nodata_value = src1.nodata
        valid_mask = (raster1 != nodata_value) & (raster2 != nodata_value) & (raster3 != nodata_value)
        
        # Compute the percentage of differing pixels
        total_valid_pixels = np.sum(valid_mask)
        
        differing_pixels_1_2 = np.sum((raster1 != raster2) & valid_mask)
        differing_pixels_1_3 = np.sum((raster1 != raster3) & valid_mask)
        differing_pixels_2_3 = np.sum((raster2 != raster3) & valid_mask)

        # Total valid pixels
        total_valid_pixels = np.sum(valid_mask)

        # Calculate percentage differences
        diff_percentage_1_2 = (differing_pixels_1_2 / total_valid_pixels) * 100 if total_valid_pixels > 0 else 0
        diff_percentage_1_3 = (differing_pixels_1_3 / total_valid_pixels) * 100 if total_valid_pixels > 0 else 0
        diff_percentage_2_3 = (differing_pixels_2_3 / total_valid_pixels) * 100 if total_valid_pixels > 0 else 0

        # Store results
        diff_results[raster_name] = {
            "Closest vs Previous": diff_percentage_1_2,
            "Closest vs Previous Year": diff_percentage_1_3,
            "Previous vs Previous Year": diff_percentage_2_3
        }

        # Print results
        print(f"{raster_name}:")
        print(f"  Closest vs Previous: {diff_percentage_1_2:.2f}% different pixels")
        print(f"  Closest vs Previous Year: {diff_percentage_1_3:.2f}% different pixels")
        print(f"  Previous vs Previous Year: {diff_percentage_2_3:.2f}% different pixels")

# Summary Output
print("\nComparison Completed. Summary of Differences:")
for raster, diffs in diff_results.items():
    print(f"{raster}: Closest vs Previous: {diffs['Closest vs Previous']:.2f}%, "
          f"Closest vs Previous Year: {diffs['Closest vs Previous Year']:.2f}%, "
          f"Previous vs Previous Year: {diffs['Previous vs Previous Year']:.2f}%")

In [ ]:
### Calculating the overall differences using all the years ############
# Initialize cumulative difference counters
# Get the list of raster files from the first folder
raster_files = sorted([f for f in os.listdir(product_paths[0]) if f.endswith(".tif")])

# Initialize difference counters for each dataset pair
total_differing_pixels_1_2 = 0  # Closest GE vs Previous GE
total_differing_pixels_1_3 = 0  # Closest GE vs Previous Year GE
total_differing_pixels_2_3 = 0  # Previous GE vs Previous Year GE

total_valid_pixels = 0  # Total valid pixel count (same for all pairs)

# Loop through each raster file
for raster_name in raster_files:
    raster_paths = [os.path.join(folder, raster_name) for folder in product_paths]

    # Open the three rasters
    with rasterio.open(raster_paths[0]) as src1, \
         rasterio.open(raster_paths[1]) as src2, \
         rasterio.open(raster_paths[2]) as src3:
        
        # Read raster data as numpy arrays
        raster1 = src1.read(1)  # Closest GE
        raster2 = src2.read(1)  # Previous GE
        raster3 = src3.read(1)  # Previous Year GE

        # Ensure all rasters have the same shape
        if not (raster1.shape == raster2.shape == raster3.shape):
            print(f"Shape mismatch for {raster_name}, skipping...")
            continue

        # Get NoData value
        nodata_value = src1.nodata

        # Generate a single valid mask for all three images
        valid_mask = (raster1 != nodata_value) & (raster2 != nodata_value) & (raster3 != nodata_value)

        # Compute differing pixels for each pair (only using valid pixels)
        differing_pixels_1_2 = np.sum((raster1 != raster2) & valid_mask)
        differing_pixels_1_3 = np.sum((raster1 != raster3) & valid_mask)
        differing_pixels_2_3 = np.sum((raster2 != raster3) & valid_mask)

        # Accumulate total differences for each dataset pair
        total_differing_pixels_1_2 += differing_pixels_1_2
        total_differing_pixels_1_3 += differing_pixels_1_3
        total_differing_pixels_2_3 += differing_pixels_2_3

        # Accumulate valid pixel count
        total_valid_pixels += np.sum(valid_mask)

# Compute total percentage difference for each dataset pair
diff_percentage_1_2 = (total_differing_pixels_1_2 / total_valid_pixels) * 100 if total_valid_pixels > 0 else 0
diff_percentage_1_3 = (total_differing_pixels_1_3 / total_valid_pixels) * 100 if total_valid_pixels > 0 else 0
diff_percentage_2_3 = (total_differing_pixels_2_3 / total_valid_pixels) * 100 if total_valid_pixels > 0 else 0

# Print final summary
print("\n=== Overall Summary of Differences Across All Time Series ===")
print(f"Total Differing Pixels (Closest GE vs Previous GE): {total_differing_pixels_1_2} ({diff_percentage_1_2:.2f}%)")
print(f"Total Differing Pixels (Closest GE vs Previous Year GE): {total_differing_pixels_1_3} ({diff_percentage_1_3:.2f}%)")
print(f"Total Differing Pixels (Previous GE vs Previous Year GE): {total_differing_pixels_2_3} ({diff_percentage_2_3:.2f}%)")
print(f"Total Valid Pixels: {total_valid_pixels}")